# Chapter 8: Scanning and Enumeration

> "If you know the enemy and know yourself, you need not fear the result of a hundred battles." -- Sun Tzu, *The Art of War*

Reconnaissance in Chapter 7 gathered intelligence from a respectful distance, mostly without touching the
target. Scanning is where the engagement becomes *active*: we send packets to the target, watch how it
responds, and from those responses build a precise picture of which hosts are alive, how the network is
shaped, which ports are open, what software is listening, which operating systems are running, and where the
exploitable weaknesses lie. This chapter turns the passive map of Chapter 7 into an actionable target list for
the exploitation work of Chapter 9.

## Learning Objectives

After completing this chapter, you will be able to:

- **Explain** why active scanning is necessary even when documentation exists, and distinguish the phases of
  scanning (host discovery, network mapping, port scanning, fingerprinting, version and vulnerability
  scanning). *(Bloom: Understand)*
- **Describe** the TCP three-way handshake, TCP control flags, and the six Nmap port states, and **predict**
  how open, closed, and filtered ports respond to each scan type. *(Bloom: Understand, Apply)*
- **Apply** Nmap and hping3 to perform host discovery, port scans, OS and version detection, and custom packet
  crafting, choosing appropriate timing and output options. *(Bloom: Apply)*
- **Analyze** scan results with a packet sniffer (Wireshark, tcpdump) to separate true findings from false
  positives. *(Bloom: Analyze)*
- **Evaluate** vulnerability-scanner output and the Nmap Scripting Engine, recognizing the limits of automated
  scanning. *(Bloom: Evaluate)*
- **Assess** detection and evasion trade-offs and conduct scanning safely, legally, and within the rules of
  engagement. *(Bloom: Evaluate)*

## Key Terms

- **Host discovery / ping sweep**: finding which IP addresses correspond to live systems.
- **Network mapping**: determining the topology (routers, firewalls, hosts) between you and the target.
- **Port scanning**: probing TCP/UDP ports to learn which are open, closed, or filtered.
- **Port state**: Nmap's classification of a port as open, closed, filtered, unfiltered, open|filtered, or closed|filtered.
- **SYN (half-open) scan**: a stealthy TCP scan that never completes the handshake.
- **Fingerprinting**: inferring the operating system or service from subtle response behavior.
- **Banner grabbing**: reading a service's self-identifying greeting to learn its software and version.
- **Enumeration**: extracting detailed information (users, shares, services) from a discovered service.
- **Version scanning**: determining the exact software/protocol version behind an open port.
- **Vulnerability scanning**: matching discovered software against known weaknesses and misconfigurations.
- **NSE**: the Nmap Scripting Engine, a Lua-based extension system for advanced probing.
- **Firewalking**: using TTL-limited packets to map what a firewall permits to the hosts behind it.


## 8.1 From Reconnaissance to Active Probing

Chapter 7 ended with an organized dossier of the target's public footprint. The natural next question is
whether all that open-source work makes scanning unnecessary, especially when the client hands us network
documentation. The answer is an emphatic no, and understanding why frames the entire chapter. **Documentation
is almost always wrong**: diagrams drift out of date, shadow IT appears, firewall rules change, and
"decommissioned" hosts keep answering. Scanning establishes the *actual* topology rather than the intended
one. Concretely, scanning lets us identify live systems and collect their IP addresses, determine system
specifics (operating system and version, open ports, running services), and surface the misconfigurations and
vulnerabilities that later phases will weaponize.

There is also a subtle but important decision at the very start: scan by **IP address, not by name**. Scanning
a hostname is problematic because organizations frequently put DNS load balancers in front of a name, so a
single hostname may resolve to many different systems. If you scan "the name," you may unknowingly probe
several machines while believing you are looking at one, your results will be inconsistent, and the
exploitation phase that follows (creating a listener, launching a specific attack) may target the wrong host
entirely. Resolve names to addresses first, confirm the scope, and scan the addresses the rules of engagement
authorize.

This chapter proceeds in the order a careful tester works: understand the scanning taxonomy and the time it
costs (8.2), refresh the TCP/IP behavior scanning depends on (8.3), discover live hosts (8.4), map the network
path (8.5), scan ports in depth with Nmap (8.6), craft custom packets with hping3 (8.7), always sniff the wire
(8.8), fingerprint operating systems (8.9), determine service versions (8.10), scan for vulnerabilities
(8.11), extend Nmap with scripts (8.12), reason about detection and evasion (8.13), and finally scan safely
and legally (8.14).


## 8.2 The Scanning Taxonomy and the Tyranny of Time

Before touching a tool, it helps to name the distinct activities lumped together as "scanning," because each
answers a different question and each has its own techniques. Building on the motivation of 8.1, we can lay out
the taxonomy as a pipeline that progressively refines our knowledge of the target.

- **Network sweeps (host discovery)** find live hosts and their IP addresses.
- **Tracing (network mapping)** discovers the topology: the routers, firewalls, and segments between us and the
  target.
- **Port scanning** determines which ports are open, closed, filtered, and so on.
- **Fingerprinting** infers the operating system, service banners, and behaviors from how the target responds.
- **Version scanning** pins down the exact version of a service or protocol.
- **Vulnerability scanning** identifies misconfigurations, vulnerable services, and missing patches.

```{mermaid}
flowchart LR
    A[Host discovery<br/>live IPs] --> B[Network mapping<br/>topology]
    B --> C[Port scanning<br/>open / closed / filtered]
    C --> D[Fingerprinting<br/>OS and behavior]
    D --> E[Version scanning<br/>exact software]
    E --> F[Vulnerability scanning<br/>weaknesses]
    F --> G[Exploitation -> Chapter 9]
```


### Why Scanning Time Dominates Planning

Scanning sounds instantaneous, but at realistic scale it is anything but, and time pressure shapes every
methodological choice that follows. Consider the arithmetic. There are 65,536 TCP ports (and the same number
of UDP ports). Suppose we must scan 500 systems and, being modest, allow one second per port:

$$(65{,}536 \text{ ports} \times 1 \text{ s}) \times 500 \text{ hosts} = 32{,}768{,}000 \text{ s} \approx 1.1 \text{ years}.$$

That is TCP alone, ignoring UDP, dropped packets, and low bandwidth, and one second per port is generous only
because real engagements sometimes demand deliberately slow, "low and slow" scans to avoid detection. Clearly
we cannot brute-force everything, so testers reduce the problem in several ways: run a **distributed scan**
from multiple sources (which can shrink a job to weeks), use **faster methods** such as lower time-out values
or tools like hping that decouple sending from receiving so the sender never waits for time-outs, take a
**representative random sample** of hosts that share configurations and services, and apply **port-scan
reduction** by scanning only the ports the customer cares about or by using customer-supplied firewall rules to
eliminate ports that are certainly blocked. These are not shortcuts so much as the difference between a project
that finishes and one that does not.

```{admonition} Knowledge Check
:class: tip
1. Why should you scan by IP address rather than by hostname?
2. Roughly how long would a naive one-second-per-port TCP scan of 500 hosts take, and name two ways to make it
   feasible.

*Answers:* (1) A hostname may resolve to many systems behind a DNS load balancer, so scanning the name yields
inconsistent results and can mislead the exploitation phase; resolve to addresses and scan authorized IPs. (2)
About 1.1 years; reduce it with distributed scanning, faster methods/lower timeouts (or hping), representative
sampling, or scanning only requested ports.
```


## 8.3 A TCP/IP Refresher for Scanners

Every scan technique is ultimately a hypothesis about how a TCP/IP stack will react to a crafted packet, so a
brief refresher on the transport layer pays for itself repeatedly. Recall from Chapter 3 that the OSI transport
layer (Layer 4) offers two very different protocols. **TCP** is connection-oriented: it maintains sequence
numbers, recovers lost packets, and establishes a connection before data flows. **UDP** is connectionless: its
packets are independent, there are no control bits and no sequence tracking, so the application is responsible
for any reliability. This difference is why UDP scanning is *conceptually* simpler (there is no connection
state to track) yet in practice slower and less reliable, as we will see.

TCP coordinates connections using **control bits (flags)** in its header, chiefly SYN (synchronize), ACK
(acknowledge), FIN (finish), RST (reset), PSH (push), and URG (urgent). A normal connection opens with the
famous **three-way handshake**: the client sends SYN, the server replies SYN-ACK, and the client completes with
ACK. Scanners exploit the fact that a stack's response to *unusual* flag combinations reveals whether a port is
open.

```{mermaid}
sequenceDiagram
    participant C as Client (scanner)
    participant S as Server (target)
    C->>S: SYN (seq=x)
    S->>C: SYN-ACK (seq=y, ack=x+1)
    C->>S: ACK (ack=y+1)
    Note over C,S: Connection established (then data, or RST to tear down)
```


### The Six Nmap Port States

Nmap does not report a binary open/closed; it distinguishes **six states**, and reading them correctly is the
single most important skill in port scanning because each encodes what the network actually did with your
probe.

| State | Meaning |
|---|---|
| **open** | An application is accepting TCP connections, UDP datagrams, or SCTP associations on the port. |
| **closed** | The port is reachable, but no application is listening (a TCP probe gets a RST). |
| **filtered** | Nmap cannot tell whether the port is open because packet filtering (firewall, router ACL, host firewall) drops the probe; an ICMP type 3 code 13 (administratively prohibited) may be returned. |
| **unfiltered** | The port is reachable but Nmap cannot tell open from closed; only the ACK scan, used to map firewall rules, yields this. |
| **open\|filtered** | Nmap cannot decide between open and filtered; open ports that simply drop probes (give no response) produce this. |
| **closed\|filtered** | Nmap cannot decide whether the port is closed or filtered. |

The distinction between *closed* and *filtered* is what separates a useful report from a misleading one. A
closed port actively refuses (a RST arrives, so the host is up and the port simply has no service); a filtered
port is silence imposed by a security device. The speed consequence is real: when responses come back as RSTs
or ICMP port-unreachable messages, scanning moves quickly because each probe resolves immediately, but when
packets are silently dropped we must wait for time-outs to expire, which dominates scan time. Some Linux
kernels (2.4 and later) further **throttle** ICMP responses to one unreachable message per second, stretching
UDP scans dramatically.


## 8.4 Host Discovery and Network Sweeps

With the transport-layer behavior in mind, the first practical step against a range is to find which addresses
are actually alive, so we do not waste the time budget of 8.2 on empty space. The classic tool is the **ping
sweep**: send an ICMP Echo Request (type 8) to each address and treat an ICMP Echo Reply (type 0) as proof of
life. Plain `ping` is fine for a handful of hosts but hopeless at scale (a Class C holds 254 usable addresses,
a Class B over 65,000), and, more importantly, **ICMP echo is very often disabled**, so absence of a reply is
not absence of a host.

Faster and more flexible sweepers solve both problems. **fping** pings many hosts in parallel and reports only
the live ones, and it can read targets from a file, which makes it ideal for scripting. Nmap performs the same
job with its host-discovery mode and a range of probe types chosen to match what a network permits.


In [ ]:
# Host discovery examples (illustrative -- run only against authorized targets)

# fping: only show alive hosts, read targets from a file
#   fping -a -f targets.txt 2>/dev/null

# Nmap ping sweep (no port scan), no DNS resolution, save live hosts to XML
#   nmap -sn -n -oX live-out.xml 10.0.0.0/24

# Extract just the live IPs from grepable output
#   nmap -sn -n -oG - 10.0.0.0/24 | awk '/Up$/{print $2}' > livehosts.txt

# Choose the discovery probe based on what the network allows:
#   -PE   ICMP Echo Request (type 8)
#   -PP   ICMP Timestamp Request
#   -PM   ICMP Address Mask Request
#   -PS22,80,443   TCP SYN to these ports
#   -PA80          TCP ACK to port 80
#   -PR   ARP ping (only works on the local subnet, but very fast and reliable)
#   -Pn   skip host discovery entirely (treat all hosts as up)
print("Host discovery is about choosing a probe the target's filters will actually answer.")

### What Nmap Probes by Default

It is worth knowing what Nmap does *before* it scans, because those default probes themselves generate traffic
a defender can see. As a privileged user (UID 0), when the target is on the **same subnet** Nmap sends an ARP
request (fast and unspoofable on a LAN); on a **different subnet** it combines an ICMP Echo Request, a TCP SYN
to port 443, a TCP ACK to port 80, and an ICMP Timestamp request. Unprivileged users fall back to a full TCP
three-way handshake to ports 80 and 443. The `-Pn` option (formerly `-P0`) skips this discovery step and is
essential when a target blocks pings but you know it is up. Understanding these defaults is what lets you both
scan effectively and, later, recognize a scan when you are defending.


## 8.5 Network Mapping: Traceroute, Firewalking, and Nmap

Knowing which hosts are alive is not the same as knowing how to reach them or what sits in the way, so the next
step is to map the path. The standard process moves from DNS to ping to traceroute to firewalking, and the goal
is to understand the topology well enough to choose the best attack route: where the openings are, what to
avoid (firewalls, tarpits, load balancers, honeypots), which systems are alive, and how addressing is laid
out.

**Traceroute** (by Van Jacobson on Unix, `tracert` on Windows) reveals the route and per-hop latency by
exploiting the IP **time-to-live (TTL)** field. It sends packets with TTL = 1, then 2, then 3, and so on; each
router that decrements the TTL to zero returns an ICMP "time exceeded" message, revealing itself, until the
destination is reached and returns an ICMP Echo Reply. A hop that does not answer is shown as `*` (filtered),
and on Unix traceroute defaults to UDP starting at port 33434 (measuring each hop three times), while Windows
`tracert` uses ICMP Echo Requests by default.

```{mermaid}
flowchart LR
    S[Scanner] -->|TTL=1| R1[Router 1]
    R1 -->|ICMP time exceeded| S
    S -->|TTL=2| R2[Router 2]
    R2 -->|ICMP time exceeded| S
    S -->|TTL=3| T[Target]
    T -->|ICMP echo reply| S
```


### Firewalking, LFT, and Nmap's Smarter Traceroute

Traceroute can be turned into a reconnaissance weapon. **Firewalking** (developed by Mike Schiffman and David
Goldsmith) is a TTL-based technique for discovering what a firewall *permits*: you send TCP or UDP packets with
the TTL set to one hop *beyond* the firewall, and if a packet passes the gateway it reaches the next hop where
TTL expires and triggers a "TTL exceeded in transit" message, revealing that the firewall forwarded that
traffic. By probing many ports this way, you map the firewall's allowed services while disguising the activity
as ordinary tracerouting. **LFT (Layer Four Traceroute)** extends the idea with a fast, multi-protocol engine
that performs autonomous-system lookups against the regional internet registries and uses loose source routing,
firewall, and load-balancer detection; it is now bundled into many traceroute distributions.

Nmap's own `--traceroute` is cleverer still. Rather than counting up from TTL = 1 for every target, it works
*backward* from the target using a TTL learned from an earlier scan, adjusting up or down until it finds the
right value, then decrementing to map each hop. When it moves to the next target it reuses the router hops it
already discovered, so mapping a second and third host is far faster than starting over. This efficiency is why
mapping is usually folded into the same Nmap invocation as scanning.


## 8.6 Port Scanning with Nmap

With live hosts found and the path mapped, we reach the heart of the chapter: determining, port by port, what
each host exposes. **Nmap** ("Network Mapper"), written by Gordon Lyon (Fyodor) and freely available at
nmap.org, is the de-facto standard. It began as a port scanner and grew to perform host discovery, OS
fingerprinting, version detection, scripting, and even vulnerability scanning; a GUI front-end, Zenmap, is
available for those who prefer it. Nmap scans the **top 1,000 ports** for TCP/UDP by default, not all 65,535
and not even all ports below 1,024, which is a deliberate balance of coverage against the time problem of 8.2.

The most common everyday targets a tester checks first are the perennially popular services: 80/http, 443/https,
22/ssh, 25/smtp, 21/ftp, 23/telnet, 53/domain, 3389/ms-term-server, 110/pop3, and 143/imap, among others.


### TCP Scan Types

Nmap exposes a family of TCP scans, each setting different control bits to elicit different behavior. The two
you will use most are the connect scan and the SYN scan.

- **TCP connect scan (`-sT`)** completes the full three-way handshake and then resets the connection. It needs
  no special privileges (it uses the operating system's `connect()` call), so any user can run it, but it is
  slower and far more likely to be logged because a full connection is established.
- **SYN (half-open / stealth) scan (`-sS`)** sends a SYN and interprets the reply: SYN-ACK means open, RST
  means closed, no response means filtered. Because it never completes the handshake, it is usually not logged
  by the application, though firewalls and intrusion-detection systems may still notice. It requires root
  privileges. This is the default scan for privileged users and the workhorse of professional testing.

Beyond these, Nmap offers scans that abuse the TCP specification to probe quietly or to map firewalls:

- **ACK scan (`-sA`)** is used to map firewall rules (it distinguishes filtered from unfiltered) rather than to
  find open ports.
- **FIN (`-sF`)**, **Null (`-sN`)**, and **Xmas (`-sX`)** scans rely on RFC 793 behavior: a closed port that
  receives a TCP packet lacking SYN, RST, and ACK must reply with RST, while an open port simply drops it. The
  FIN scan sets only FIN, the Null scan sets no flags, and the Xmas scan sets FIN, PSH, and URG (lighting up
  the packet "like a Christmas tree"). Because not every OS obeys the RFC, these scans also aid OS
  fingerprinting.
- **Maimon scan (`-sM`)** sets FIN and ACK.
- **Custom flags (`--scanflags`)** let you set any combination yourself, for example
  `nmap --scanflags SYNPSHURG -p 80 10.0.0.5`.


In [ ]:
# Nmap scan-type and option examples (illustrative -- authorized targets only)

# SYN (stealth) scan of common ports, no DNS, with service versions and OS detection
#   sudo nmap -sS -n -sV -O -p 1-1024 10.0.0.5

# Connect scan (no root needed)
#   nmap -sT 10.0.0.5

# UDP scan of the most common UDP services (slow!)
#   sudo nmap -sU --top-ports 50 10.0.0.5

# Firewall-rule mapping with an ACK scan
#   sudo nmap -sA 10.0.0.5

# Aggressive scan (-A = OS + version + default scripts + traceroute)
#   sudo nmap -A 10.0.0.5

# Specify exact ports and protocols; -r turns OFF random port order
#   nmap -p T:22,80,443,U:53,161 -r 10.0.0.5

# Fast scan of top 100 ports
#   nmap -F 10.0.0.5
print("Pick the scan type by privilege level, stealth needs, and whether you are finding ports or mapping a firewall.")

### UDP Scanning, Timing, and Output

UDP scanning (`-sU`) deserves special care. With no connection state, scanning is conceptually easier, but it
is slow and less reliable: open ports often give no response at all, and modern Nmap sends protocol-specific
payloads to a handful of well-known UDP ports (for example 53/domain, 161/snmp, 123/ntp, 500/isakmp) to coax a
reply, leaving others with a blank payload. Combined with the one-ICMP-per-second throttling mentioned in 8.3,
a full 65,535-port UDP scan of a single host can take eighteen hours or more, which is why testers scan only
the UDP ports that matter.

**Timing** is controlled with the `-T` templates, a crucial lever for balancing speed against stealth and
accuracy:

| Template | Name | Behavior |
|---|---|---|
| `-T0` | paranoid | ~5 minutes between packets (serial); used to evade IDS |
| `-T1` | sneaky | ~15 seconds between packets (serial) |
| `-T2` | polite | ~0.4 seconds between packets (serial) |
| `-T3` | normal | default (parallel) |
| `-T4` | aggressive | max 1.25 s wait for a response |
| `-T5` | insane | max 0.3 s wait; fastest, most likely to miss results |

Finally, capture your results. Nmap writes **normal** output with `-oN`, **grepable** with `-oG`, **XML** with
`-oX`, and all three at once with `-oA basename`. XML is the format to keep: it feeds reporting pipelines, and
`ndiff scan1.xml scan2.xml` shows exactly what changed between two scans, while `xsltproc nmap.xml -o nmap.html`
renders a readable report. While a scan runs you can press keys for live feedback: `v`/`Shift+v` adjust
verbosity, `p`/`Shift+p` toggle packet tracing, `d`/`Shift+d` adjust debugging, and any other key prints a
status line with percent complete and time remaining.

```{admonition} In-Class Exercise: read the port states
:class: note
On an authorized lab host, run `sudo nmap -sS -p 22,80,443,8080 -T4 <target>` and record the state of each
port. Then add `--reason` and re-run. For each port, write down the *evidence* Nmap used (syn-ack, reset,
no-response, or an ICMP message) and explain why that evidence maps to the state shown. Finally, re-run against
a host behind a firewall and contrast which ports become `filtered`.
```


## 8.7 hping3: Crafting Packets by Hand

Nmap automates common cases, but sometimes you need to build a packet bit by bit, to test a firewall rule,
spoof a source, or send traffic faster than a polite scanner would. That is the role of **hping3**, a
command-line TCP/IP packet assembler and analyzer (by Salvatore Sanfilippo) freely available at hping.org.
Beyond ICMP echo (like `ping`), it speaks TCP, UDP, and RAW-IP, has a traceroute mode, supports Tcl scripting,
and is used for firewall testing, advanced port scanning, remote OS fingerprinting, and uptime guessing. By
default it sends a TCP packet with no control bits set.

Because hping separates sending from receiving, it does not wait for time-outs, which makes it fast and is one
of the "faster methods" mentioned in 8.2. You set control bits explicitly with `--syn`, `--ack`, `--fin`,
`--rst`, `--urg`, `--push`, and you can set TTLs, IP IDs, checksums, and TCP sequence numbers by hand.


In [ ]:
# hping3 examples (illustrative -- authorized targets only; can DoS stateful firewalls)

# Send a single UDP packet
#   hping3 --udp 192.168.10.10

# SYN to port 80, one packet
#   hping3 --syn -p 80 --count 1 192.168.10.10

# Spoof the source address (firewall/route testing)
#   hping3 -a 192.168.10.50 10.10.10.10

# Sweep a /24 by random last octet on a chosen interface
#   hping3 --rand-dest 192.168.10.x -I eth0

# Iterate a real IP range with a shell loop, keep only lines showing a live "ip="
#   for x in $(seq 100 200); do hping3 --count 1 192.168.10.${x} 2>/dev/null | grep "ip="; done

# Heartbeat: one ICMP echo every 5 seconds with an audible beep
#   hping3 --icmp --interval 5 --beep 192.168.10.10

# Speed controls: --fast (~10/s), --faster (~100/s), --flood (as fast as possible, no replies shown)
print("hping3 trades convenience for total control of every header field.")

```{admonition} In-Class Exercise: see a spoofed packet
:class: note
Open two terminals on an authorized lab network. In the first, run `sudo tcpdump -Xn not arp`. In the second,
run `sudo hping3 -a 192.168.10.55 10.10.10.150 --count 1`. In the tcpdump output, identify the source and
destination IP addresses and confirm the source is the *spoofed* address you supplied, not your real one.
Discuss why spoofing the source means you will not see the replies, and why flooding SYNs at a target risks a
denial of service from half-open connections.
```

```{admonition} Warning: packet crafting can cause outages
:class: warning
hping3 with `--flood`, `--rand-dest`, or many `--syn` packets can overwhelm stateful firewalls and crash
fragile services, producing an accidental denial of service. Spoofing and flooding must be explicitly permitted
in the rules of engagement (8.14). Power tools demand explicit authorization.
```


## 8.8 Always Sniff the Wire

A rule that separates professionals from script-runners follows directly from the packet-level thinking of the
last two sections: **always run a sniffer while you scan.** A capture lets you confirm the scanner is actually
working, identify false positives, and review the exact packets sent and received when results look wrong, and
it is invaluable evidence during an incident-response assessment. You do not have to save every packet (full
captures consume space, though they are wise during IR), but you should at least watch the traffic on screen.

The two essential tools are **Wireshark**, a graphical protocol analyzer that captures and interactively
browses live traffic, reads and writes dozens of capture formats, decrypts many protocols (IPsec, Kerberos,
SSL/TLS, WPA/WPA2 with keys), applies coloring rules, and exports to CSV, XML, or text; and **tcpdump**, the
lightweight command-line capturer (run as root) with a powerful filter language. Two companion web resources
are worth bookmarking: ExplainShell, which breaks down what a command and its flags do, and ShellCheck, which
finds bugs in the Bash you write to drive your scans.


In [ ]:
# tcpdump examples (illustrative)

# All TCP to a host, no name resolution
#   sudo tcpdump -n tcp and dst host 192.168.10.10

# TCP to port 80 of a host, in hex and ASCII
#   sudo tcpdump -nX tcp and port 80 and dst 192.168.10.10

# All UDP between two specific hosts
#   sudo tcpdump udp and src 10.10.10.10 and dst 192.168.10.10

# Everything to/from one host
#   sudo tcpdump -n host 192.168.10.10

# Useful flags: -A (ASCII) -X (hex+ASCII) -n (no name lookup) -i <iface> -w file.pcap (write) -r file.pcap (read)
print("If the sniffer disagrees with the scanner, believe the sniffer.")

## 8.9 Operating-System Fingerprinting

Once ports are mapped, narrowing the target's operating system sharpens every later decision, because an
exploit for Windows 7 will not work on Windows XP SP2, and a Linux daemon behaves differently from its BSD
cousin. Fingerprinting comes in two flavors.

**Active fingerprinting** sends a battery of carefully chosen probes and measures the responses, exploiting the
fact that different OS TCP/IP, ICMP, and other protocol implementations behave differently in their corner
cases. Nmap's `-O` option does exactly this, comparing responses against a large signature database, and the
RFC-violating FIN/Null/Xmas scans of 8.6 contribute clues because not all stacks obey the specification.
**Xprobe2** is a dedicated active fingerprinter built on Ofir Arkin's ICMP fingerprinting research that uses
fuzzy matching to estimate the OS.

**Passive fingerprinting** never sends a packet of its own; it merely observes traffic. **p0f** (by Michal
Zalewski) is the classic passive tool: by inspecting the characteristics of ordinary TCP connections it can
identify the OS and software on both endpoints, detect NAT and load balancing, and even spot clients that forge
declarative fields such as the HTTP `User-Agent` or email `X-Mailer` header. Passive fingerprinting is
invaluable when stealth is paramount, since it adds no traffic for a defender to notice.


## 8.10 Service and Version Scanning, and Enumeration

Knowing a port is open and guessing the OS is not enough; we need the *exact* software and version behind each
port, because vulnerabilities are version-specific. **Version scanning** with Nmap's `-sV` connects to open
ports and matches the service's responses against a database of probes and signatures, reporting, for example,
not merely "80/open" but "Apache httpd 2.4.49." A simpler manual cousin is **banner grabbing**, reading the
greeting a service volunteers when you connect (with `netcat`, `telnet`, or an Nmap script). Versions let us
narrow the search for known weaknesses, though we must remember that a version known to be vulnerable may still
be protected by other controls, so a positive finding is a lead, not a conclusion.

**Enumeration** goes deeper into specific services to extract usable detail. A few high-value examples recur in
nearly every engagement:

- **SMB enumeration** (Windows file sharing, ports 139/445) can reveal shares, users, groups, and policies via
  tools like `enum4linux`, `smbclient`, and Nmap's `smb-*` scripts.
- **SNMP enumeration** (UDP 161) is a goldmine when default community strings such as `public` are left in
  place, exposing interfaces, routes, ARP tables, and running processes via `snmpwalk` or `onesixtyone`.
- **Web content discovery** (directory brute-forcing) with `gobuster`, `feroxbuster`, or `dirb` uncovers hidden
  paths, admin panels, and backup files that the site never links to.


In [ ]:
# Version detection and enumeration examples (illustrative -- authorized only)

# Service/version detection plus default scripts
#   sudo nmap -sV -sC -p- 10.0.0.5

# Banner grab with netcat
#   nc -nv 10.0.0.5 22

# SMB enumeration
#   enum4linux -a 10.0.0.5
#   nmap --script "smb-enum-shares,smb-enum-users" -p445 10.0.0.5

# SNMP with the default community string
#   snmpwalk -v2c -c public 10.0.0.5
#   onesixtyone -c communities.txt 10.0.0.5

# Web directory brute force
#   gobuster dir -u http://10.0.0.5 -w /usr/share/wordlists/dirb/common.txt
print("Version + enumeration turn an open port into a concrete attack hypothesis.")

## 8.11 Vulnerability Scanning

Having identified operating systems, services, and versions, the next step is to match them against known
weaknesses, and here automated **vulnerability scanners** earn their keep. Once we know the OS and version we
can look up software vulnerabilities, starting with the OS but extending to every application, and where we
cannot read a version we can sometimes infer a flaw from *behavior* that is consistent with a known
vulnerability, or, if we already have access, by checking the system for misconfigurations.

The best-known commercial scanner is **Nessus** (Tenable), with **OpenVAS/Greenbone** as the leading
open-source alternative and Qualys and Rapid7 InsightVM in the enterprise space. A scanner is driven by
**policies** that set technical parameters (timeouts, host counts, port-scanner type), credentials for
authenticated scans (Windows, SSH, database, web), and plugin selection. Default policies typically include an
external network scan (all 65,535 ports plus web-app checks on internet-facing hosts), an internal network scan
(tuned for many hosts and embedded systems like printers), a web-application test (spidering and fuzzing), and
a PCI DSS audit comparison.

Two operational cautions matter. First, **update plugins before every engagement** and record the plugin-feed
version and your configuration, because results are only as current as the signatures. Second, **dangerous
plugins exist**: some checks send malformed packets or denial-of-service probes that can crash a target (the
DoS category is partly enabled by default), so you must consciously decide whether to enable them and confirm
the client accepts the risk.

```{admonition} Warning: a scan is a start, not an answer
:class: warning
Vulnerability scanners are limited. They take the services you discovered and look for *evidence* of known
weaknesses, but they cannot hold an adaptive, multi-step conversation with a service to confirm exploitability,
and they produce false positives and false negatives. Treat scanner output as leads to verify by hand (and with
tools like the Nmap Scripting Engine), never as a finished assessment.
```


## 8.12 The Nmap Scripting Engine (NSE)

The gap between a raw vulnerability scanner and a thinking tester is partly filled by the **Nmap Scripting
Engine (NSE)**, which lets Nmap run small programs that probe far more intelligently than a single packet can.
NSE scripts are written in **Lua** (a fast, free, embeddable language also used in games) and can send and
receive arbitrary messages to many targets in parallel, perform extended discovery (WHOIS, DNS), do advanced
version detection, conduct vulnerability scanning, detect backdoored systems, and even exploit discovered
flaws.

Scripts are organized into **categories** (such as `safe`, `default`, `discovery`, `intrusive`, `vuln`,
`exploit`, `auth`, `brute`, and `malware`) and indexed in `scripts/script.db`. NSE recognizes four script
phases: **prerule** scripts (before scanning), **host** scripts, **service** scripts, and **postrule** scripts
(after scanning). You run the default set with `-sC` (equivalent to `--script default`), a category or named
script with `--script <name>`, and add `--script-trace` for detailed output. Treat the `all` category with
respect: it can include exploit and denial-of-service scripts.


In [ ]:
# NSE examples (illustrative -- authorized targets only)

# Run the default safe scripts during a scan
#   nmap -sC -p 80,443 10.0.0.5

# Run a category of scripts (e.g., vulnerability checks)
#   nmap --script vuln 10.0.0.5

# Run a specific script with arguments
#   nmap --script http-title -p 80 10.0.0.5
#   nmap --script dns-zone-transfer --script-args dns-zone-transfer.domain=example.com -p53 ns.example.com

# Find scripts by category in the script database
#   cat /usr/share/nmap/scripts/script.db | grep safe | wc -l
print("NSE turns Nmap from a port scanner into a lightweight, extensible probing framework.")

## 8.13 Detection and Evasion

Every technique in this chapter generates traffic, and defenders watch for it, so a professional must
understand both how scans are detected and how (within authorization) they can be made quieter. Intrusion
detection systems flag scanning by recognizing its patterns: many connection attempts from one source in a
short window, probes that walk sequentially or randomly across ports, packets with illegal flag combinations
(Null, FIN, Xmas), and bursts that exceed normal rates. The Nmap timing templates of 8.6 exist largely to
defeat this: a `-T0` paranoid serial scan spaces packets minutes apart precisely so the activity blends into
the background noise rather than spiking a threshold.

Several lower-level techniques further complicate detection, and each has a defensive lesson:

- **Fragmentation (`-f`)** splits probes across multiple IP fragments so a simple signature does not match,
  though modern sensors reassemble.
- **Decoys (`-D`)** mix your real source address among many spoofed ones so the defender cannot easily tell who
  is really scanning.
- **Source-port manipulation and timing** (`--source-port`, slow templates) disguise probes as benign traffic.
- **Bad checksums (`--badsum`)** are a clever trick called "firewall spotting": well-behaved hosts silently
  drop packets with invalid Layer-4 checksums, so *any* reply must have come from a firewall or IDS that did
  not validate the checksum, revealing the security device itself. Comparing the TTL of SYN-ACK responses for
  allowed services against the TTL of RST responses for blocked ones can likewise expose filtering.
- **IPv6 (`-6`)** is frequently left unsecured: firewalls and IDS that carefully police IPv4 may ignore IPv6,
  which is auto-configured on modern Windows, Linux, and macOS, so an IPv6 scan can slip past controls that
  would catch the IPv4 equivalent.

The defender's takeaways mirror the attacker's tricks: rate-limit and correlate connection attempts, alert on
illegal flag combinations, reassemble fragments, validate checksums, and apply the same filtering rules to IPv6
as to IPv4.


## 8.14 Scanning Safely, Legally, and Within Scope

The power demonstrated throughout this chapter is exactly why it must be wielded carefully. Unauthorized
scanning is, in many jurisdictions, a criminal act under computer-misuse laws (Chapter 18), and even authorized
scanning can cause harm: aggressive timing, flooding, fragmentation, spoofing, and dangerous vulnerability
plugins can crash fragile services, trip a stateful firewall into a denial of service, or generate alerts that
disrupt operations. Three disciplines keep scanning professional.

First, **operate strictly within the rules of engagement**: scan only the addresses, ports, time windows, and
techniques the written authorization permits, and confirm whether spoofing, flooding, and DoS-category checks
are allowed before using them. Second, **escalate carefully**: begin with light, polite scans and increase
intensity only as needed, watching the target's health. Third, **document everything**: keep packet captures
and Nmap XML output (8.6, 8.8) so findings are reproducible and so you can prove what you did and did not do if
a system misbehaves during the engagement. If connection replies suddenly stop, you may have been "shunned" by
a defensive device, which itself is information, but verify from another source carefully and record it, because
probing from new vantage points signals your interest to anyone watching.


<!--CH08EXTRA-->
### Automated Discovery and Mapping Tools

Manual traceroute and Nmap mapping scale poorly across large enterprises, so dedicated discovery products
automate topology construction, and a tester should know they exist both to use them and to recognize their
footprints. **SolarWinds Network Topology Mapper** automates device discovery, builds multiple maps from a
single scan, exports diagrams to Visio, performs multi-level discovery, and auto-detects topology changes.
**Open-AudIT** is an open-source tool that scans the network and stores configuration details about every
discovered device. **PRTG Network Monitor** (Paessler) draws custom diagrams (sunbursts, device trees, global
and Layer-2 maps) and monitors continuously. These tools trade the surgical control of Nmap and hping3 for
breadth and repeatability, which is exactly what an internal asset inventory needs; an attacker who compromises
such a console inherits a ready-made map of the whole estate, which is why monitoring servers are
high-value targets in their own right.


### Scanning IPv6

A blind spot that recurs in real assessments deserves its own treatment: **IPv6 is frequently left
unsecured.** It auto-configures on modern Windows, Linux, and macOS, yet firewalls and intrusion-detection
systems that meticulously police IPv4 often ignore it, so an IPv6 path can bypass controls that would stop the
equivalent IPv4 scan. IPv6 addresses are 128 bits (16 bytes), written as eight groups of hex digits, with a
double colon `::` standing in once for a run of zero groups (the loopback is `::1`). Because the address space
is astronomically large, brute-force sweeps are infeasible, so discovery relies on **Neighbor Discovery**
instead: a multicast ping to the all-routers address `ff02::1` and the all-nodes address `ff02::2` elicits
responses, after which `ip neigh` lists discovered neighbors. A specific target is written with its scope
(zone) identifier, for example `fe80::20c0%eth0`, and then scanned with Nmap's `-6` option.


In [ ]:
# IPv6 discovery and scanning (illustrative -- authorized only)

# Discover link-local neighbors via multicast
#   ping6 -I eth0 ff02::1     # all nodes
#   ping6 -I eth0 ff02::2     # all routers
#   ip neigh                  # list discovered neighbors

# Scan a scoped IPv6 target with version detection and a packet trace
#   sudo nmap -6 -Pn -sV fe80::20c0%eth0 --packet-trace

# Note: Nmap supports -6 with -sn/-sT/-sV but has limited IPv6 OS detection and no random IPv6 targets.
print("Apply the same firewall and IDS rules to IPv6 as to IPv4, or it becomes the open back door.")

### The Pentester's Toolkit: Kali and BackTrack

The tools in this chapter rarely run on a stock desktop; testers work from a purpose-built distribution that
ships them pre-installed and pre-configured. The current standard is **Kali Linux** (by Offensive Security), a
Debian-based system that bundles Nmap, hping3, Wireshark, tcpdump, the SMB/SNMP enumerators, Nessus-compatible
tooling, the Metasploit Framework, and hundreds more. Kali is the successor to **BackTrack**, the earlier live
CD that first popularized a single bootable security toolkit; the lineage matters because much older
documentation and course material (including the slide decks behind this chapter) reference BackTrack while
describing the same workflow now found in Kali. Two web utilities pair naturally with this toolkit and are
worth keeping open while you work: **ExplainShell** decodes any command and its flags, and **ShellCheck**
lints the Bash scripts you write to drive repetitive scans.

```{admonition} Knowledge Check
:class: tip
1. Why can an IPv6 scan sometimes succeed where the equivalent IPv4 scan is blocked?
2. Name one automated topology-mapping product and one reason a defender both benefits from and is endangered by
   it.
3. What is the relationship between BackTrack and Kali Linux?

*Answers:* (1) IPv6 is auto-configured on modern systems but firewalls/IDS often fail to police it as
strictly as IPv4, so it can bypass IPv4-only controls. (2) For example SolarWinds Network Topology Mapper,
Open-AudIT, or PRTG; it gives defenders an accurate live asset inventory, but an attacker who compromises the
console inherits a complete map of the network. (3) Kali Linux is the modern successor to the older BackTrack
live-CD security distribution; both bundle the same kind of pre-installed tooling.
```


## Why This Matters

Scanning is the hinge between knowing *about* a target and being able to *act* on it. Done well, it produces a
verified, prioritized inventory of reachable hosts, exposed services, exact software versions, and probable
weaknesses, the raw material every later phase depends on, while a sloppy scan wastes the time budget, misses
the one open port that mattered, or causes an outage that ends the engagement early. The same skills serve the
defender: understanding exactly how scans look on the wire is what lets a blue team detect reconnaissance,
tune sensors, and harden the very ports and protocols an attacker would probe first. Because scanning is also
the most visible and legally sensitive active step, it is where discipline, authorization, and careful
documentation matter most.


## News in Focus: Internet-Scale Scanning as an Early-Warning Signal (2025-2026)

Scanning is not only something testers do in a scoped engagement; it happens continuously across the entire
internet, and measuring it has become a defensive discipline in its own right. According to reporting on
GreyNoise Intelligence's *State of the Edge* research, the firm's sensor network observed roughly **2.97
billion sessions between July 23 and December 31, 2025** across more than **80 countries**, averaging about
**212 malicious sessions per second**, and identified **104 distinct activity surges across 18 vendors**, with
edge devices (routers, VPNs, firewalls, and other security appliances) consistently among the most targeted.

The striking finding, per the reporting, is predictive: GreyNoise found that roughly half of the scanning and
exploitation surges tracked between mid-December 2025 and late March 2026 were followed within three weeks by a
vulnerability disclosure from the targeted vendor, with a **median lead time of about nine days**, and in some
cases far longer (reported examples include exploitation of a serious Cisco flaw about 39 days before
disclosure, a VMware flaw about 36 days before, and a MikroTik flaw about 24 days before). The research also
noted that about **39% of the IP addresses targeting edge devices came from residential connections**, many
invisible to reputation feeds. The lesson for this chapter is twofold: the scanning techniques you are learning
are run at planetary scale every second, and watching that "background noise" can give defenders days of
warning before a vulnerability is even public. (Figures are per reporting on GreyNoise's data set and should be
read as the vendor's measurements.)


## Review Questions (MCQ)

1. Why is scanning by IP address preferred over scanning by hostname?
   a) Hostnames are case-sensitive
   b) A hostname may resolve to several systems behind a DNS load balancer, yielding inconsistent results
   c) Nmap cannot accept hostnames
   d) IP scanning is always stealthier

2. In a SYN (half-open) scan, what response indicates an **open** port?
   a) RST
   b) ICMP type 3 code 13
   c) SYN-ACK
   d) No response

3. Which Nmap port state means a packet filter is dropping the probe so Nmap cannot determine open vs. closed?
   a) closed
   b) unfiltered
   c) filtered
   d) open

4. What is the primary purpose of an ACK scan (`-sA`)?
   a) To complete the three-way handshake
   b) To map firewall rules (filtered vs. unfiltered)
   c) To grab service banners
   d) To detect the OS

5. Which tool is a passive OS fingerprinter that sends no packets of its own?
   a) Xprobe2
   b) hping3
   c) p0f
   d) Nessus

6. The Xmas scan sets which TCP flags?
   a) SYN only
   b) FIN, PSH, and URG
   c) SYN and ACK
   d) RST only

7. Why does the "bad checksum" technique reveal a firewall or IDS ("firewall spotting")?
   a) Firewalls always increase the TTL
   b) Well-behaved hosts drop bad-checksum packets, so any reply must come from a device that did not validate the checksum
   c) Bad checksums speed up scans
   d) Only closed ports respond to bad checksums

8. What is a major limitation of automated vulnerability scanners?
   a) They cannot read XML
   b) They produce false positives/negatives and cannot adaptively confirm exploitability
   c) They never need plugin updates
   d) They only work on UDP

9. The Nmap Scripting Engine (NSE) is written in which language?
   a) Python
   b) Ruby
   c) Lua
   d) C

10. Which Nmap timing template is the slowest and most evasive?
    a) `-T5` insane
    b) `-T3` normal
    c) `-T0` paranoid
    d) `-T4` aggressive

### Answer Key

1: **b** — DNS load balancing can map one name to many hosts. 2: **c** — SYN-ACK means open; RST means closed;
silence means filtered. 3: **c** — filtered. 4: **b** — the ACK scan maps firewall rules. 5: **c** — p0f is
passive. 6: **b** — FIN, PSH, URG. 7: **b** — only a non-validating device replies, exposing itself. 8: **b** —
scanners give leads, not proof. 9: **c** — Lua. 10: **c** — `-T0` paranoid spaces packets ~5 minutes apart.


## Hands-on Labs

These labs require an **authorized** lab environment (for example, a local VM such as Metasploitable or a
range you own). Never scan systems you do not have written permission to test.

**Lab 8.1 -- Host discovery and the time budget.** On a `/24` lab network, run a ping sweep with both
`fping -a -f targets.txt` and `nmap -sn`. Compare which hosts each finds and how long each takes. Then estimate
how long a full 65,535-port TCP scan of every live host would take at one port per second, and describe two
concrete ways you would reduce that time.

**Lab 8.2 -- Port states and scan types.** Against one authorized host, run a connect scan (`-sT`), a SYN scan
(`-sS`), and an ACK scan (`-sA`) over the same port list with `--reason`. Build a table of port -> state ->
evidence for each scan and explain the differences. Capture all three scans with `tcpdump -w lab82.pcap` and
confirm in Wireshark that the SYN scan never completes the handshake.

**Lab 8.3 -- Fingerprinting and versions.** Run `nmap -O -sV` against the target, then independently grab three
service banners with `netcat`. Note any disagreement between the version Nmap reports and the banner, and
explain why a service might misreport its version.

**Lab 8.4 -- Enumeration and NSE.** If the target exposes SMB or SNMP, enumerate it with the appropriate Nmap
scripts and a dedicated tool (`enum4linux` or `snmpwalk` with the `public` community). Then run `nmap --script
vuln` and verify by hand whether one reported issue is a true positive, documenting your reasoning.

**Lab 8.5 -- Evasion and detection (paired).** With a lab IDS (such as Snort or Suricata) running, scan once
with `-T4` and once with `-T0 -f -D RND:5`. Compare the alerts generated and write a short note on the
trade-off between stealth and scan time, and on what detection rule would still catch the slow scan.


## References

1. Lyon, G. F. (2009). *Nmap Network Scanning: The Official Nmap Project Guide to Network Discovery and Security Scanning.* Nmap Project. (Also nmap.org/book.)
2. Nmap Reference Guide. https://nmap.org/book/man.html
3. hping3 documentation. http://www.hping.org/
4. Wireshark User's Guide. https://www.wireshark.org/docs/
5. tcpdump and libpcap. https://www.tcpdump.org/
6. Schiffman, M., and Goldsmith, D. Firewalking: A Traceroute-Based Technique for Mapping Firewall Rulesets.
7. Arkin, O. ICMP Usage in Scanning (the basis of Xprobe2 fingerprinting).
8. Zalewski, M. *p0f v3: Passive OS Fingerprinting.* https://lcamtuf.coredump.cx/p0f3/
9. Tenable. *Nessus* documentation. https://docs.tenable.com/
10. Greenbone. *OpenVAS / Greenbone Vulnerability Management.* https://www.greenbone.net/
11. The PTES Technical Guidelines (Penetration Testing Execution Standard), Intelligence Gathering and Scanning sections.
12. Engebretson, P. (2013). *The Basics of Hacking and Penetration Testing* (2nd ed.). Syngress. (Scanning chapter.)
13. CyberScoop / Cybersecurity Dive reporting on GreyNoise Intelligence *State of the Edge* (2025-2026) and scanning-surge early-warning research. https://cyberscoop.com/greynoise-traffic-surge-early-warning-system-network-edge-device-vulnerabilities/
14. See also Appendix E (the author's networking and packet-analysis works) and Chapter 3 (TCP/IP, OSI) for the protocol background this chapter builds on.


```{index} Host discovery / ping sweep, Network mapping, Port scanning, Port state, SYN, Fingerprinting, Banner grabbing, Enumeration, Version scanning, Vulnerability scanning, NSE, Firewalking
```
